![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces a realistic astronomy workflow for turning a real archived sky image into an exploratory source catalogue. It covers retrieving a FITS image from an online archive, inspecting FITS metadata, displaying the field with astronomical coordinates, estimating and subtracting the background, detecting sources with <code>DAOStarFinder</code>, performing simple circular-aperture photometry, comparing analysis choices, and using interactive widgets to explore threshold and aperture effects.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Designed for students in introductory astronomy, data analysis, or scientific Python courses. Suitable for a live classroom demonstration, guided practical/lab session, or independent notebook-based practice where learners are introduced to astronomical image analysis and basic photometry in a hands-on way.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> BETA 2026<br>
    <b>Server:</b> Astronomy<br>
    <b>Kernel:</b> Python 3
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b>
    <br>
    - Internet access to retrieve a public FITS image from <code>astroquery.skyview</code> (with local caching to reuse the downloaded file)<br>
    - Real astronomical image: DSS2 Red cutout of <b>M13 (Hercules Globular Cluster)</b><br>
    - Python packages: <code>numpy</code>, <code>pandas</code>, <code>matplotlib</code>, <code>plotly</code>, <code>ipywidgets</code>, <code>aplpy</code>, <code>astroquery</code>, <code>astropy</code>, <code>photutils</code><br>
    - Standard library modules: <code>pathlib</code>, <code>warnings</code>
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 13 July 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>

# From Archive to Insight: Interactive Source Detection and Photometry in a Real Astronomical Image

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a lightweight but realistic astronomy workflow. We will download a real image of the globular cluster M13, inspect it, estimate the background, detect sources, and measure simple aperture photometry.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Archive science is a huge part of modern astronomy. A real project often begins with a public survey image, then moves toward source detection, measurement, and careful interpretation.
</div>

In this notebook, we combine:
- <b>astroquery</b> for public archive access
- <b>APLpy</b> for a sky-aware astronomical display
- <b>photutils</b> for source detection and aperture photometry
- <b>Plotly</b> for polished quantitative summaries
- <b>ipywidgets</b> for learner-controlled experimentation

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar (or use `Shift + Enter`).

In [ ]:
print("Starting setup: importing our astronomy and plotting libraries...")

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display

import aplpy
from astroquery.skyview import SkyView

# Import the missing SkyCoord class from astropy LIVE COMPATIBLE
from astropy.coordinates import SkyCoord

from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm

from photutils.background import Background2D, MedianBackground
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, aperture_photometry

warnings.filterwarnings("ignore", category=UserWarning)
plt.style.use("default")

print("Success! Imports are ready. We can now reach into a public archive and analyse a real sky image.")

## 2. Retrieving a Real Astronomical Image

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Download one compact, visually rich astronomical image that is ideal for source detection. We will use a DSS2 Red cutout centred on <b>M13</b>, the Hercules Globular Cluster.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Target context:</b> M13 is a dense ball of old stars. That makes it excellent for teaching because it is bright, crowded, and scientifically realistic: some stars are easy to spot, while the crowded centre reminds us that source detection is never perfect.
</div>

The next cell downloads the image once, saves it locally, and reuses the cached FITS file if you rerun the notebook.

In [ ]:
print("Starting data retrieval: requesting a real sky image from SkyView...")

# EDIT THIS VALUE: choose a human-readable label for the notebook
target_name = "M13 (Hercules Globular Cluster)"

# EDIT THIS VALUE: fixed sky coordinates for reliability
target_position = "16h41m41.24s +36d27m35.5s"

# EDIT THIS VALUE: choose a public survey available through SkyView
survey_name = "DSS2 Red"

# EDIT THIS VALUE: radius of the image cutout
radius_arcmin = 12

# EDIT THIS VALUE: output image size in pixels (square)
image_pixels = 600

data_dir = Path("astronomy_demo_data")
data_dir.mkdir(exist_ok=True)
fits_path = data_dir / "m13_dss2_red_12arcmin.fits"

if fits_path.exists():
    print(f"Using cached FITS file: {fits_path}")
    hdul = fits.open(fits_path)
else:
    try:
        # Explicitly define the coordinate frame to fix the warning
        coord = SkyCoord(target_position, frame='icrs')
        
        images = SkyView.get_images(
            position=coord,
            survey=[survey_name],
            radius=radius_arcmin * u.arcmin,
            pixels=image_pixels,
        )
    except Exception as exc:
        raise RuntimeError("SkyView retrieval failed. Please rerun the cell or check network access.") from exc

    if len(images) == 0:
        raise RuntimeError("SkyView returned no image for this request.")

    hdul = images[0]
    hdul.writeto(fits_path, overwrite=True)
    print(f"Downloaded and saved FITS file: {fits_path}")

data = np.squeeze(hdul[0].data.astype(float))
header = hdul[0].header
wcs = WCS(header)
hdul.close()

data = np.nan_to_num(data, nan=np.nanmedian(data))
ny, nx = data.shape
pixel_scale_arcsec = abs(header.get("CDELT1", np.nan)) * 3600
field_width_arcmin = (nx * pixel_scale_arcsec / 60) if np.isfinite(pixel_scale_arcsec) else np.nan

print(f"Image ready: {ny} x {nx} pixels")
print(f"Approximate pixel scale: {pixel_scale_arcsec:.2f} arcsec/pixel")
print(f"Approximate field width: {field_width_arcmin:.1f} arcmin")

## 3. Inspecting the Field and Its Metadata

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Look at the image metadata before doing science. This helps us understand what the image is, how big it is, and what coordinate system it uses.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Good habit:</b> Astronomy analysis should be reproducible. Image headers tell us where the image came from, what sky coordinates it uses, and how pixel positions map back onto the sky.
</div>

In [ ]:
print("Extracting a compact metadata summary from the FITS header...")

metadata_rows = [
    ("Target label", target_name),
    ("Requested sky position", target_position),
    ("Survey", survey_name),
    ("Image shape", f"{ny} x {nx} pixels"),
    ("Approx. pixel scale", f"{pixel_scale_arcsec:.2f} arcsec/pixel"),
    ("Approx. field width", f"{field_width_arcmin:.1f} arcmin"),
    ("Reference RA", f"{header.get('CRVAL1', np.nan):.6f} deg"),
    ("Reference Dec", f"{header.get('CRVAL2', np.nan):.6f} deg"),
    ("Longitude axis type", header.get("CTYPE1", "Not available")),
    ("Latitude axis type", header.get("CTYPE2", "Not available")),
    ("Brightness unit", header.get("BUNIT", "Instrumental / not provided")),
    ("Source file", str(fits_path)),
]

metadata_df = pd.DataFrame(metadata_rows, columns=["Property", "Value"])
display(metadata_df)

print("Metadata ready: we know the image scale, sky reference point, and where the FITS file is stored.")

## 4. First Look at the Sky Field

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Make a sky-aware display of the FITS image. We want to see the field as astronomers usually do: with brightness stretch, celestial axes, and a sense of scale.
</div>

Run the next cell and take a moment to inspect the image. The bright, crowded centre is one of the reasons M13 is such a good teaching target.

In [ ]:
print("Rendering the FITS image with APLpy...")

# Hide the astropy deprecation warning
warnings.filterwarnings("ignore", message=".*CoordinateHelper.ticks.*")

fig = plt.figure(figsize=(7.5, 7.5))
fits_figure = aplpy.FITSFigure(str(fits_path), figure=fig)
fits_figure.show_grayscale(stretch="asinh", invert=False, pmin=1, pmax=99.7)
fits_figure.add_grid()
fits_figure.grid.set_alpha(0.25)
fits_figure.grid.set_color("white")
fits_figure.add_scalebar(2 / 60)
fits_figure.scalebar.set_label("2 arcmin")
fits_figure.scalebar.set_color("white")
fits_figure.axis_labels.set_xtext("Right Ascension")
fits_figure.axis_labels.set_ytext("Declination")
plt.show()

print("Sky-aware image ready. Notice the dense core and the more isolated sources around the edges.")

## 5. Estimating the Background

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Background estimation:</b> Real images are not just stars. They also contain sky background, detector effects, and large-scale variations. If we want to detect sources fairly, we should estimate and subtract that background first.
</div>

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a background model, subtract it, and estimate a robust noise level. That noise estimate will help us define a sensible detection threshold in units of sigma.
</div>

In [ ]:
print("Estimating the background and noise level...")

# EDIT THIS VALUE: larger boxes track only broad background structure
background_box_size = 40

background = Background2D(
    data,
    box_size=(background_box_size, background_box_size),
    filter_size=(3, 3),
    bkg_estimator=MedianBackground(),
)

data_sub = data - background.background
mean_clip, median_clip, rms_clip = sigma_clipped_stats(data_sub, sigma=3.0)

print(f"Sigma-clipped mean after subtraction: {mean_clip:.2f}")
print(f"Sigma-clipped median after subtraction: {median_clip:.2f}")
print(f"Estimated RMS noise: {rms_clip:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(data, origin="lower", cmap="gray", norm=simple_norm(data, "asinh", percent=99.7))
axes[0].set_title("Original image")

axes[1].imshow(background.background, origin="lower", cmap="magma")
axes[1].set_title("Estimated background")

axes[2].imshow(data_sub, origin="lower", cmap="gray", norm=simple_norm(data_sub, "asinh", percent=99.7))
axes[2].set_title("Background-subtracted image")

for ax in axes:
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")

plt.tight_layout()
plt.show()

print("Background estimation complete. The right-hand panel is the version we will analyse.")

## 6. Detecting Sources and Measuring Simple Photometry

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Detect the brightest point-like sources and measure a simple brightness value for each one using circular aperture photometry.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Aperture photometry:</b> We place a circular aperture around each detected source and sum the light inside it. This gives an <b>instrumental flux</b>, not a calibrated physical magnitude. It is still very useful for comparison inside one image.
</div>

We will use a default detection threshold of <b>5 sigma</b> and a default aperture radius of <b>4 pixels</b>. Later, you will compare different choices.

In [ ]:
print("Preparing source-detection and photometry tools...")

def detect_sources_table(image_subtracted, rms_value, threshold_sigma=5.0, fwhm_px=3.0):
    finder = DAOStarFinder(
        threshold=threshold_sigma * rms_value,
        fwhm=fwhm_px,
        exclude_border=True
    )
    detections = finder(image_subtracted)

    if detections is None or len(detections) == 0:
        return pd.DataFrame(columns=["xcentroid", "ycentroid", "peak", "flux"])

    df = detections.to_pandas()
    df.columns = [str(c).strip() for c in df.columns]

    # Robustly map possible centroid column names to standard ones
    rename_map = {}

    if "xcentroid" not in df.columns:
        for candidate in ["xcentroid", "x_centroid", "xcenter", "x_center", "xcen"]:
            if candidate in df.columns:
                rename_map[candidate] = "xcentroid"
                break

    if "ycentroid" not in df.columns:
        for candidate in ["ycentroid", "y_centroid", "ycenter", "y_center", "ycen"]:
            if candidate in df.columns:
                rename_map[candidate] = "ycentroid"
                break

    df = df.rename(columns=rename_map)

    required_cols = ["xcentroid", "ycentroid", "peak", "flux"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(
            f"Missing expected detection columns: {missing_cols}. "
            f"Available columns are: {list(df.columns)}"
        )

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=required_cols)
    df = df[df["flux"] > 0].copy()
    df = df.sort_values("flux", ascending=False).reset_index(drop=True)
    return df


def measure_aperture_photometry(image_subtracted, sources_df, aperture_radius_px=4.0):
    if sources_df.empty:
        result = sources_df.copy()
        result["aperture_radius_px"] = pd.Series(dtype=float)
        result["aperture_sum"] = pd.Series(dtype=float)
        return result

    positions = np.column_stack((
        sources_df["xcentroid"].to_numpy(),
        sources_df["ycentroid"].to_numpy()
    ))
    apertures = CircularAperture(positions, r=aperture_radius_px)
    phot_table = aperture_photometry(image_subtracted, apertures)
    aperture_sum = np.asarray(phot_table["aperture_sum"], dtype=float)

    result = sources_df.copy()
    result["aperture_radius_px"] = aperture_radius_px
    result["aperture_sum"] = aperture_sum
    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["aperture_sum"])
    result = result[result["aperture_sum"] > 0].copy()
    result = result.sort_values("aperture_sum", ascending=False).reset_index(drop=True)
    return result


def add_world_coordinates(sources_df, current_wcs):
    if sources_df.empty:
        result = sources_df.copy()
        result["ra_deg"] = pd.Series(dtype=float)
        result["dec_deg"] = pd.Series(dtype=float)
        return result

    ra_deg, dec_deg = current_wcs.all_pix2world(
        sources_df["xcentroid"].to_numpy(),
        sources_df["ycentroid"].to_numpy(),
        0,
    )

    result = sources_df.copy()
    result["ra_deg"] = ra_deg
    result["dec_deg"] = dec_deg
    return result


# EDIT THIS VALUE: detection threshold in multiples of the image noise
default_threshold_sigma = 5.0

# EDIT THIS VALUE: approximate stellar width in pixels for DAOStarFinder
default_fwhm_px = 3.0

# EDIT THIS VALUE: circular aperture radius for brightness measurement
default_aperture_radius_px = 4.0

catalogue = detect_sources_table(
    data_sub,
    rms_value=rms_clip,
    threshold_sigma=default_threshold_sigma,
    fwhm_px=default_fwhm_px,
)

catalogue = measure_aperture_photometry(
    data_sub,
    catalogue,
    aperture_radius_px=default_aperture_radius_px,
)

catalogue = add_world_coordinates(catalogue, wcs)
catalogue["source_rank"] = np.arange(1, len(catalogue) + 1)
catalogue["distance_from_center_px"] = np.hypot(
    catalogue["xcentroid"] - nx / 2,
    catalogue["ycentroid"] - ny / 2
)

if len(catalogue) > 0:
    catalogue["log10_aperture_sum"] = np.log10(catalogue["aperture_sum"])
    brightest_ap_sum = catalogue["aperture_sum"].max()
else:
    catalogue["log10_aperture_sum"] = pd.Series(dtype=float)
    brightest_ap_sum = np.nan

print(f"Detection complete: {len(catalogue)} sources found at {default_threshold_sigma:.1f}σ.")
if np.isfinite(brightest_ap_sum):
    print(f"Brightest measured aperture sum: {brightest_ap_sum:.1f}")
else:
    print("Brightest measured aperture sum: no valid sources detected.")

display(
    catalogue[
        ["source_rank", "xcentroid", "ycentroid", "ra_deg", "dec_deg", "peak", "aperture_sum"]
    ].head(12).round(3)
)

print("Catalogue preview ready. These are instrumental measurements from this image, not calibrated magnitudes.")

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> If you want to experiment immediately, go back to the previous cell and try changing the detection threshold from <b>5.0</b> to <b>4.0</b> or <b>6.0</b>. Also try changing the aperture radius from <b>4</b> to <b>3</b> or <b>5</b> pixels. Then rerun the cell and compare the results.
</div>

In [ ]:
print("Overlaying the detected sources on the astronomical image...")

# EDIT THIS VALUE: limit the overlay for readability
n_to_overlay = min(150, len(catalogue))

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(data, origin="lower", cmap="gray", norm=simple_norm(data, "asinh", percent=99.7))
ax.scatter(
    catalogue["xcentroid"].head(n_to_overlay),
    catalogue["ycentroid"].head(n_to_overlay),
    s=50,
    facecolors="none",
    edgecolors="cyan",
    linewidths=0.8,
    alpha=0.8,
)

for _, row in catalogue.head(10).iterrows():
    ax.text(row["xcentroid"] + 3, row["ycentroid"] + 3, int(row["source_rank"]), color="yellow", fontsize=8)

ax.set_title(f"Top {n_to_overlay} detected sources overlaid on the DSS2 image")
ax.set_xlabel("x pixel")
ax.set_ylabel("y pixel")
plt.show()

print("Overlay ready. The numbered labels mark the ten brightest sources in our catalogue.")

## 7. Comparing Analysis Choices

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Compare how our scientific choices change the results. We will look at two very common decisions: the <b>detection threshold</b> and the <b>aperture radius</b> used for photometry.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Scientific idea:</b> Lower thresholds usually detect more sources, but they may include more false positives. Larger apertures usually capture more light, but they also collect more contamination from neighbours and background.
</div>

In [ ]:
print("Comparing several source-detection thresholds...")

threshold_grid = [4.0, 5.0, 6.0, 7.0]
threshold_rows = []

for threshold_sigma in threshold_grid:
    tmp_catalogue = detect_sources_table(
        data_sub,
        rms_value=rms_clip,
        threshold_sigma=threshold_sigma,
        fwhm_px=default_fwhm_px,
    )
    tmp_catalogue = measure_aperture_photometry(
        data_sub,
        tmp_catalogue,
        aperture_radius_px=default_aperture_radius_px,
    )

    threshold_rows.append(
        {
            "threshold_sigma": threshold_sigma,
            "n_sources": len(tmp_catalogue),
            "median_top20_aperture_sum": float(tmp_catalogue["aperture_sum"].head(20).median()) if len(tmp_catalogue) > 0 else np.nan,
        }
    )

threshold_summary = pd.DataFrame(threshold_rows)
display(threshold_summary)

fig = px.line(
    threshold_summary,
    x="threshold_sigma",
    y="n_sources",
    markers=True,
    title="How the detection threshold changes the number of sources",
    labels={"threshold_sigma": "Detection threshold (sigma)", "n_sources": "Detected sources"},
)
fig.update_traces(line=dict(color="#4C72B0", width=3), marker=dict(size=10))
fig.update_layout(template="plotly_white")
fig.show()

print("Threshold comparison ready. Notice how stricter thresholds usually reduce the source count.")

In [ ]:
print("Comparing two aperture sizes on the same bright sources...")

# Use the brightest sources from the baseline catalogue for a clean comparison
comparison_subset = catalogue.head(40).copy()

phot_r3 = measure_aperture_photometry(data_sub, comparison_subset, aperture_radius_px=3.0)
phot_r5 = measure_aperture_photometry(data_sub, comparison_subset, aperture_radius_px=5.0)

aperture_comparison = pd.DataFrame(
    {
        "source_rank": comparison_subset["source_rank"].to_numpy(),
        "flux_r3": phot_r3["aperture_sum"].to_numpy(),
        "flux_r5": phot_r5["aperture_sum"].to_numpy(),
    }
)
aperture_comparison["flux_ratio_r5_over_r3"] = aperture_comparison["flux_r5"] / aperture_comparison["flux_r3"]

display(aperture_comparison.head(10).round(3))

max_flux = float(np.nanmax(aperture_comparison[["flux_r3", "flux_r5"]].to_numpy()))
median_ratio = float(np.nanmedian(aperture_comparison["flux_ratio_r5_over_r3"]))

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=aperture_comparison["flux_r3"],
        y=aperture_comparison["flux_r5"],
        mode="markers",
        marker=dict(size=9, color="#55A868", opacity=0.8),
        text=[f"Source {rank}" for rank in aperture_comparison["source_rank"]],
        hovertemplate="%{text}<br>r=3 px: %{x:.1f}<br>r=5 px: %{y:.1f}<extra></extra>",
        name="Measured sources",
    )
)
fig.add_trace(
    go.Scatter(
        x=[0, max_flux],
        y=[0, max_flux],
        mode="lines",
        line=dict(color="black", dash="dash"),
        name="Equal flux line",
    )
)
fig.update_layout(
    title="Larger apertures usually capture more light",
    xaxis_title="Aperture sum with radius = 3 px",
    yaxis_title="Aperture sum with radius = 5 px",
    template="plotly_white",
)
fig.show()

print(f"Median flux ratio (5 px / 3 px): {median_ratio:.2f}")
print("Aperture comparison ready. This is why photometry settings should always be reported clearly.")

## 8. Interactive Explorer

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> This is where the workflow becomes exploratory. You can change the detection threshold and aperture radius, then immediately see how the number of detections and the brightness distribution respond.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try this:</b> Start with <b>5σ</b> and <b>4 px</b>. Then compare it with <b>4σ</b>, and finally with <b>7σ</b>. Watch what happens to the crowded core and to the histogram of measured brightness values.
</div>

Run the next cell, then use the sliders.

In [ ]:
print("Building the interactive source-detection explorer...")

def explore_detection(threshold_sigma=5.0, aperture_radius=4, brightest_n=30):
    trial = detect_sources_table(
        data_sub,
        rms_value=rms_clip,
        threshold_sigma=threshold_sigma,
        fwhm_px=default_fwhm_px,
    )
    trial = measure_aperture_photometry(
        data_sub,
        trial,
        aperture_radius_px=aperture_radius,
    )

    if trial.empty:
        print("No sources were detected with these settings. Try lowering the threshold.")
        return

    trial = trial.sort_values("aperture_sum", ascending=False).reset_index(drop=True)
    trial["source_rank"] = np.arange(1, len(trial) + 1)
    top_trial = trial.head(brightest_n)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    axes[0].imshow(data, origin="lower", cmap="gray", norm=simple_norm(data, "asinh", percent=99.7))
    axes[0].scatter(
        top_trial["xcentroid"],
        top_trial["ycentroid"],
        s=(aperture_radius * 12),
        facecolors="none",
        edgecolors="lime",
        linewidths=0.8,
    )
    axes[0].set_title(f"Top {len(top_trial)} detections at {threshold_sigma:.1f}σ")
    axes[0].set_xlabel("x pixel")
    axes[0].set_ylabel("y pixel")

    axes[1].hist(np.log10(trial["aperture_sum"]), bins=20, color="#4C72B0", alpha=0.85)
    axes[1].set_title("Brightness distribution")
    axes[1].set_xlabel("log10(aperture sum)")
    axes[1].set_ylabel("Number of sources")

    plt.tight_layout()
    plt.show()

    print(
        f"Detected {len(trial)} sources | Median aperture sum = {trial['aperture_sum'].median():.1f} | Brightest displayed = {len(top_trial)}"
    )


widgets.interact(
    explore_detection,
    threshold_sigma=widgets.FloatSlider(
        value=default_threshold_sigma,
        min=3.5,
        max=8.0,
        step=0.5,
        description="Threshold σ:",
        continuous_update=False,
    ),
    aperture_radius=widgets.IntSlider(
        value=int(default_aperture_radius_px),
        min=2,
        max=6,
        step=1,
        description="Aperture r:",
        continuous_update=False,
    ),
    brightest_n=widgets.IntSlider(
        value=30,
        min=10,
        max=100,
        step=10,
        description="Brightest N:",
        continuous_update=False,
    ),
);

print("Interactive explorer ready. Try moving one control at a time so the effect is easy to interpret.")

## 9. Responsible Interpretation and Workflow Limits

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important caution:</b> This notebook demonstrates a sensible teaching workflow, but it does <b>not</b> produce a publication-ready stellar catalogue. Crowding, saturation, blending, and aperture choices all affect the measurements.
</div>

A few important limitations to keep in mind:

- <b>Crowding:</b> in the dense cluster core, neighbouring stars overlap. Some detections may actually contain blended light from more than one star.
- <b>Saturation and plate effects:</b> bright sources and photographic survey data can behave differently from modern CCD survey images.
- <b>Threshold dependence:</b> a lower threshold finds more candidates, but some may be noise peaks or imperfectly separated sources.
- <b>Aperture dependence:</b> larger apertures capture more flux, but also more contamination from neighbours and residual background.
- <b>No calibration step:</b> our aperture sums are instrumental values. Turning them into calibrated magnitudes would require additional information and corrections.
- <b>Completeness:</b> the faintest stars are not all detected. We should not treat the measured catalogue as the full stellar population of M13.

That is normal in astronomy. Good science is not about pretending a workflow is perfect — it is about understanding what the workflow can and cannot support.

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have taken a real astronomical image from archive retrieval to a first scientific catalogue. You inspected FITS metadata, displayed the sky field, estimated the background, detected sources, measured simple aperture photometry, compared analysis choices, and explored the workflow interactively.
</div>

Strong next steps could include:
- trying a different target field or survey
- cross-matching detections with an external catalogue
- estimating local rather than global backgrounds for each star
- using source segmentation or PSF-based photometry for crowded fields
- comparing this historical survey image with a modern survey cutout

This is a realistic example of how astronomy turns archived observations into quantitative insight.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)